# Prompt Chaining II

# Task
รับคำถามจากผู้ใช้งาน ซึ่งเป็นคำถามที่ต้องการค้นหาข้อมูลจากฐานข้อมูล (Database) จากนั้นให้ AI Agent ทำงานตามขั้นตอนเพื่อค้นหาคำตอบหรือสร้างกราฟที่เหมาะสม

การทำงานของ Agent:
Agent จะต้องแยกประเภทคำถาม (Classify) ออกเป็น 2 ประเภท และทำงานตาม Flow ดังต่อไปนี้:

## 1. Text Question
ลักษณะคำถาม: เป็นคำถามตรงไปตรงมาที่ผู้ใช้คาดหวังคำตอบสั้นๆ เช่น ตัวเลข 1-2 จำนวน หรือประโยคคำตอบสั้นๆ

#### กระบวนการทำงาน:

1. แปลงคำถามของผู้ใช้ให้เป็นคำสั่ง SQL โดยอ้างอิงจากโครงสร้างฐานข้อมูล (DB Schema)

2. ตรวจสอบความถูกต้องของ SQL (SQL Validation): ตรวจสอบว่าคำสั่ง SQL ถูกต้องหรือไม่ หากไม่ถูกต้อง ระบบจะส่งฟีดแบ็ก (Feedback) กลับไปให้แก้ไขและสร้าง SQL ใหม่จนกว่าจะถูกต้อง

3. รันคำสั่ง SQL (Execute) เพื่อดึงข้อมูลจากฐานข้อมูล

4. นำผลลัพธ์ที่ได้มาเรียบเรียงเป็นคำตอบภาษาธรรมชาติ (Natural language) ที่อ่านเข้าใจง่าย

**ตัวอย่างคำถาม**: "ใครมีจำนวนอัลบั้มมากที่สุดในปี 2020 ?" (Who has the most number of album in 2020?)

##  2. Plot Question 
ลักษณะคำถาม: เป็นคำถามที่ผู้ใช้ต้องการเห็นภาพรวม การเปรียบเทียบ แนวโน้ม หรือการกระจายตัวของข้อมูล ซึ่งต้องอาศัยการแสดงผลแบบกราฟ (Visualization) เพื่อให้เข้าใจได้ง่ายขึ้น

#### กระบวนการทำงาน:

1. แปลงคำถามให้เป็นคำสั่ง SQL ที่สามารถดึงข้อมูลออกมาเป็นชุด (Multi-row data / Time-series) ที่เหมาะสำหรับการสร้างกราฟ

2. ตรวจสอบความถูกต้องของ SQL (SQL Validation): เช่นเดียวกับแบบแรก หากผิดพลาดให้วนลูปกลับไปแก้ไข

3. รันคำสั่ง SQL (Execute) เพื่อดึงข้อมูลจากฐานข้อมูล

4. นำข้อมูลผลลัพธ์ที่ได้มาสร้างเป็นโค้ดกราฟในรูปแบบ Mermaid.js

5. ตรวจสอบความถูกต้องของกราฟ (Mermaid Validation): ให้ LLM ตรวจสอบว่าโค้ด Mermaid ถูกต้องตามหลักไวยากรณ์ (Syntax) หรือไม่ และเช็คว่าข้อมูลในกราฟสามารถตอบคำถามของผู้ใช้ได้จริงหรือไม่

    - หากถูกต้อง: แสดงผลกราฟให้ผู้ใช้

    - หากไม่ถูกต้อง: (เช่น ข้อมูลไม่พอสร้างกราฟ, โค้ดผิด) ระบบจะนำฟีดแบ็กที่ได้ วนลูปกลับไปเริ่มต้นสร้างคำสั่ง SQL ใหม่ เพื่อดึงข้อมูลให้ถูกต้องตั้งแต่ต้น

**ตัวอย่างคำถาม**: "ใครมีจำนวนอัลบั้มมากที่สุดในแต่ละปี และมีจำนวนเท่าไร ?" (Who has the most number of album each year and by how many? plot a graph)


## STEP1: Design

In [99]:
from IPython.display import Image
from IPython.core.display import HTML 
Image(url= "./Examples/prompt-chain-iii.png", width=700)

In [100]:
# !uv pip install langchain-ollama langgraph pydantic

In [101]:
# !uv pip install mermaid-py


## STEP2: Build Graph

In [102]:
from typing import Annotated, TypedDict, Optional, Literal, Tuple
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END

In [103]:
class AgentState(TypedDict):
    # TODO
    pass

class QueryClassification(BaseModel):
    query_type: Literal["text question", "plot question"] = Field(...)

class SQLQuery(BaseModel):
    sql_query: str = Field(description="The valid SQL query.")

class MermaidValidationResult(BaseModel):
    is_valid: bool = Field(description="True if the mermaid code is syntactically correct and accurately answers the user's question based on the data. False otherwise.")
    feedback: str = Field(description="If invalid, provide specific feedback on what is wrong. If valid, return 'Valid'.")

class MermaidOutput(BaseModel):
    mermaid_code: str = Field(description="The raw mermaid.js code.")
    

In [104]:
llm = ChatOllama(model="scb10x/typhoon2.5-qwen3-4b")
classifier_llm = llm.with_structured_output(QueryClassification)
sql_llm = llm.with_structured_output(SQLQuery)
mermaid_llm = llm.with_structured_output(MermaidOutput)
mermaid_validator_llm = llm.with_structured_output(MermaidValidationResult)

### External Tools

In [105]:
import mermaid as md
from mermaid.graph import Graph

def render_mermaid(code):
    return md.Mermaid(code)
    
# render_mermaid("""
# stateDiagram-v2
#     [*] --> Still
#     Still --> [*]

#     Still --> Moving
#     Moving --> Still
#     Moving --> Crash
#     Crash --> [*]
# """)

In [106]:
sqlSchema = '''
CREATE TABLE "artists"
(
    [ArtistId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);

CREATE TABLE "albums"
(
    [AlbumId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Title] NVARCHAR(160)  NOT NULL,
    [ArtistId] INTEGER  NOT NULL,
    FOREIGN KEY ([ArtistId]) REFERENCES "artists" ([ArtistId]) 
        ON DELETE NO ACTION ON UPDATE NO ACTION
);

CREATE TABLE "genres"
(
    [GenreId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);

CREATE TABLE "media_types"
(
    [MediaTypeId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);

CREATE TABLE "tracks"
(
    [TrackId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(200)  NOT NULL,
    [AlbumId] INTEGER,
    [MediaTypeId] INTEGER  NOT NULL,
    [GenreId] INTEGER,
    [Composer] NVARCHAR(220),
    [Milliseconds] INTEGER  NOT NULL,
    [Bytes] INTEGER,
    [UnitPrice] NUMERIC(10,2)  NOT NULL,
    FOREIGN KEY ([AlbumId]) REFERENCES "albums" ([AlbumId]) 
        ON DELETE NO ACTION ON UPDATE NO ACTION,
    FOREIGN KEY ([GenreId]) REFERENCES "genres" ([GenreId]) 
        ON DELETE NO ACTION ON UPDATE NO ACTION,
    FOREIGN KEY ([MediaTypeId]) REFERENCES "media_types" ([MediaTypeId]) 
        ON DELETE NO ACTION ON UPDATE NO ACTION
);
'''

In [107]:
import sqlite3
import os

def execute_query(query):
    """
    Connects to a SQLite database, executes a given query, and returns the results.

    Args:
        query (str): The SQL query string to execute.

    Returns:
        list: A list of tuples, where each tuple represents a row from the results.
              Returns None if an error occurs.
    """
    connection = None
    results = None
    
    try:
        db_file = "./Examples/example.db"
        connection = sqlite3.connect(db_file)
        cursor = connection.cursor()
        
        print(f"Executing query:\n{query}\n")
        cursor.execute(query)
        results = cursor.fetchall()
        
    except sqlite3.Error as e:
        print(f"An error occurred: {e}")
        
    finally:
        if connection:
            connection.close()
            
    return results

In [108]:
execute_query("SELECT * FROM albums LIMIT 5")

Executing query:
SELECT * FROM albums LIMIT 5



[(1, 'For Those About To Rock We Salute You', 1),
 (2, 'Balls to the Wall', 2),
 (3, 'Restless and Wild', 2),
 (4, 'Let There Be Rock', 1),
 (5, 'Big Ones', 3)]

In [109]:
def sql_validation(query: str) -> bool:
    """
    Validates if the given SQL query is syntactically correct without actually fetching or modifying data.
    
    Args:
        query (str): The SQL query string to validate.
    """
    try:
        db_file = "./Examples/example.db"
        conn = sqlite3.connect(db_file)
        cursor = conn.cursor()
        cursor.execute(f"EXPLAIN {query}")
        conn.close()
        return True, None
    except Exception as e:
        return False, str(e)

In [110]:
sql_validation("SELECT * FROM albums LIMIT 5")

(True, None)

In [111]:
sql_validation("SELECT * FROM albumsx LIMIT 5")

(False, 'no such table: albumsx')

### Define Nodes

In [169]:
def ClassifierNode(state: AgentState):
    # TODO
    pass

In [170]:
def TextSQLGeneratorNode(state: AgentState):
    # TODO
    pass

In [171]:
def PlotSQLGeneratorNode(state: AgentState):
    # TODO
    pass

In [172]:
def SQLValidatorNode(state: AgentState):
    # TODO
    pass

def SQLExecutorNode(state: AgentState):
    # TODO
    pass

def ResponseGeneratorNode(state: AgentState):
    # TODO
    pass

In [173]:
import json

def MermaidGeneratorNode(state: AgentState):
    # TODO
    pass

def MermaidValidatorNode(state: AgentState):
    # TODO
    pass

### Define Edges & Logic

In [175]:
def route_after_classifier(state: AgentState):
    print("Route [route_after_classifier]: ", state.get("query_type"))
    return "TextSQLGeneratorNode" if state.get("query_type") == "text question" else "PlotSQLGeneratorNode"

def route_simple_sql_validation(state: AgentState):
    print("Route [route_simple_sql_validation]: ", state.get("is_sql_valid"))
    return "TextSQLExecutorNode" if state.get("is_sql_valid") else "TextSQLGeneratorNode"

def route_plot_sql_validation(state: AgentState):
    print("Route [route_plot_sql_validation]: ", state.get("is_sql_valid"))
    return "PlotSQLExecutorNode" if state.get("is_sql_valid") else "PlotSQLGeneratorNode"

def route_mermaid_validation(state: AgentState):
    print("Route [route_mermaid_validation]: ", state.get("is_mermaid_valid"))
    return END if state.get("is_mermaid_valid") else "PlotSQLGeneratorNode"

### Define the graph

In [176]:
workflow = StateGraph(AgentState)

# Add Nodes
workflow.add_node("ClassifierNode", ClassifierNode)
workflow.add_node("TextSQLGeneratorNode", TextSQLGeneratorNode)
workflow.add_node("PlotSQLGeneratorNode", PlotSQLGeneratorNode)
workflow.add_node("TextSQLValidatorNode", SQLValidatorNode)
workflow.add_node("PlotSQLValidatorNode", SQLValidatorNode)
workflow.add_node("TextSQLExecutorNode", SQLExecutorNode)
workflow.add_node("PlotSQLExecutorNode", SQLExecutorNode)
workflow.add_node("ResponseGeneratorNode", ResponseGeneratorNode)
workflow.add_node("MermaidGeneratorNode", MermaidGeneratorNode)
workflow.add_node("MermaidValidatorNode", MermaidValidatorNode)

# Top level routing
workflow.add_edge(START, "ClassifierNode")
workflow.add_conditional_edges("ClassifierNode", route_after_classifier, {
    "TextSQLGeneratorNode": "TextSQLGeneratorNode",
    "PlotSQLGeneratorNode": "PlotSQLGeneratorNode"
})

# Simple Flow (With Validation Loop)
workflow.add_edge("TextSQLGeneratorNode", "TextSQLValidatorNode")
workflow.add_conditional_edges("TextSQLValidatorNode", route_simple_sql_validation, {
    "TextSQLExecutorNode": "TextSQLExecutorNode",
    "TextSQLGeneratorNode": "TextSQLGeneratorNode"
})
workflow.add_edge("TextSQLExecutorNode", "ResponseGeneratorNode")
workflow.add_edge("ResponseGeneratorNode", END)

# Plot Flow (With SQL AND Mermaid Validation Loops)
workflow.add_edge("PlotSQLGeneratorNode", "PlotSQLValidatorNode")
workflow.add_conditional_edges("PlotSQLValidatorNode", route_plot_sql_validation, {
    "PlotSQLExecutorNode": "PlotSQLExecutorNode",
    "PlotSQLGeneratorNode": "PlotSQLGeneratorNode"
})
workflow.add_edge("PlotSQLExecutorNode", "MermaidGeneratorNode")
workflow.add_edge("MermaidGeneratorNode", "MermaidValidatorNode")

# Mermaid Reflection Loop back to SQL Generation
workflow.add_conditional_edges("MermaidValidatorNode", route_mermaid_validation, {
    END: END,
    "PlotSQLGeneratorNode": "PlotSQLGeneratorNode"
})

app = workflow.compile()

In [177]:
"DONE"

'DONE'

### Try

In [182]:
from IPython.display import Image, display

user_input = {"messages": [HumanMessage(content="How many albums are in the database?")]}
result = app.invoke(user_input)
current_state = result

if current_state["query_type"]=="plot question":
    print("HERE are the plot:")
    try:
        display(render_mermaid(current_state["mermaid_code"]))
    except Exception as e:
        print("Cannot plot", str(e))
else:
    print(f"HERE are the answer: {current_state['messages'][-1].content}")


In [183]:
from IPython.display import Image, display

user_input = {"messages": [HumanMessage(content="Plot number of albums by the artists")]}
result = app.invoke(user_input)
current_state = result

if current_state["query_type"]=="plot question":
    print("HERE are the plot:")
    try:
        display(render_mermaid(current_state["mermaid_code"]))
    except Exception as e:
        print("Cannot plot", str(e))
else:
    print(f"HERE are the answer: {current_state['messages'][-1].content}")
